In [1]:
#import libraries, data

import numpy as np
import pandas as pd
data = pd.read_csv('archive/mnist_train.csv')
print(data.shape)
print(data.head())

(60000, 785)
   label  1x1  1x2  1x3  1x4  1x5  1x6  1x7  1x8  1x9  ...  28x19  28x20  \
0      5    0    0    0    0    0    0    0    0    0  ...      0      0   
1      0    0    0    0    0    0    0    0    0    0  ...      0      0   
2      4    0    0    0    0    0    0    0    0    0  ...      0      0   
3      1    0    0    0    0    0    0    0    0    0  ...      0      0   
4      9    0    0    0    0    0    0    0    0    0  ...      0      0   

   28x21  28x22  28x23  28x24  28x25  28x26  28x27  28x28  
0      0      0      0      0      0      0      0      0  
1      0      0      0      0      0      0      0      0  
2      0      0      0      0      0      0      0      0  
3      0      0      0      0      0      0      0      0  
4      0      0      0      0      0      0      0      0  

[5 rows x 785 columns]


In [2]:
#data organisation and arrangement

data = np.array(data)
X = data[:, 1:].T/255
Y = data[:, 0]
dev = X[:, :10000]
train = X[:, 10000:60000]
dev_y = Y[:10000]
train_y = Y[10000:60000]

In [3]:
#initializing weights and biases, using np.random.randn()

W1 = np.random.randn(128, 784) * 0.01
W2 = np.random.randn(10, 128) * 0.01
b1 = np.zeros((128,1))
b2 = np.zeros((10, 1))

In [4]:
#defining ReLU, softmax, logits ,and activations

def relu (x):
    return np.maximum(0, x)

def softmax (x):
    x = x - np.max(x, axis = 0, keepdims = True)
    return np.exp(x) / np.sum(np.exp(x), axis = 0, keepdims = True)


#forward pass

z1 = np.dot(W1, train) + b1
a1 = relu(z1)
z2 = np.dot(W2, a1) + b2
a2 = softmax(z2)

#cost function

cost = -np.sum(np.log(a2[train_y.astype(int), np.arange(50000)])) / 50000

In [5]:
#using one hot encoding

def one_hot(Y) :
    one_hot_Y = np.zeros((10, Y.size))
    one_hot_Y[Y.astype(int), np.arange(Y.size)] = 1
    return one_hot_Y

Y_onehot = one_hot(train_y)

In [6]:
#backward propagation

dz2 = a2 - Y_onehot
dW2 = (1/50000) * dz2@a1.T
db2 = (1/50000) * np.sum(dz2, axis = 1, keepdims = True)
da1 = W2.T @ dz2
dz1 = da1 * (z1 > 0)
dw1 = (1/50000) * dz1 @ train.T
db1 = (1/50000) * np.sum(dz1, axis = 1, keepdims = True)

In [7]:
#updating the weights and biases, and initalizing the learning factor alpha

alpha = 0.5
W1 -= alpha * dw1
b1 -= alpha * db1
W2 -= alpha * dW2
b2 -= alpha * db2

In [8]:
#training loop ~ 500 iterations

for i in range(500) :
    
    #forward pass
    z1 = np.dot(W1, train) + b1
    a1 = relu(z1)
    z2 = np.dot(W2, a1) + b2
    a2 = softmax(z2)
    
    cost = -np.sum(np.log(a2[train_y.astype(int), np.arange(50000)])) / 50000
    if i % 100 == 0 : print(cost)
    
    #backward pass
    dz2 = a2 - Y_onehot
    dW2 = (1/50000) * dz2@a1.T
    db2 = (1/50000) * np.sum(dz2, axis = 1, keepdims = True)
    da1 = W2.T @ dz2
    dz1 = da1 * (z1 > 0)
    dw1 = (1/50000) * dz1 @ train.T
    db1 = (1/50000) * np.sum(dz1, axis = 1, keepdims = True)
    
    #updates
    alpha = 0.5
    W1 -= alpha * dw1
    b1 -= alpha * db1
    W2 -= alpha * dW2
    b2 -= alpha * db2
    
    
    
    

2.297467577086483
0.3595398593527781
0.2554256741854046
0.2125091012212032
0.1818828495022595


In [9]:
#testing and accuracy check

z1 = np.dot(W1, dev) + b1
a1 = relu(z1)
z2 = np.dot(W2, a1) + b2
a2 = softmax(z2)

predictions = np.argmax(a2, axis = 0)
accuracy = np.mean(predictions ==dev_y)
print(accuracy)

0.9495
